# PRÁCTICA 8.2 : WRANGLER

In [175]:
import pandas as pd  # Para trabajar con datos tabulares.
import re  # Para manejar patrones de texto con expresiones regulares.

In [177]:
# Rutas de los archivos de entrada y salida.
input_csv = "universities_data.csv"  # Archivo original generado por crawler.py.
output_csv = "universities_data_cleaned.csv"  # Archivo con los datos limpios.

# Cargamos el archivo CSV en un DataFrame.
df = pd.read_csv(input_csv)

## Consideración inicial 

Para datos faltantes, seguiremos el siguiente criterio:

1.- Si el dato faltante es de **tipo int**, lo fijaremos en **None**, ya que puede facilitar futuras operaciones sobre ellos.

2.- Si el dato faltante es de **tipo string**, le asignaremos **"No applicable"**, a modo informativo para el usuario.

___

## 8.2.1. LIMPIEZA DE LA VARIABLE TYPE (TIPO DE UNIVERSIDAD)

In [182]:
def clean_type_column(type_value):
    """
    Limpia y formatea los valores de la columna 'type'.

    Tareas realizadas:
    1. Separa palabras pegadas: 'Privateresearchuniversity' -> 'Private Research University'.
    2. Convierte todas las palabras a "Title Case" (primera letra mayúscula).
    3. Asegura que palabras específicas como 'state' estén correctamente capitalizadas.

    Input:
        type_value (str): El valor de la columna 'type' que se va a limpiar.
                          Puede ser una cadena o un valor nulo (None/NaN) (protección frente a errores, aunque en 
                          principio los Null's han sido sustituidos en el Crawler por "No applicable"). En ese caso devolveremos None.

    Output:
        str | None: El valor limpio y formateado. Si el input es nulo, retorna None.
    """

    # Si el valor está vacío o es nulo, devolvemos None.
    if pd.isnull(type_value):
        return None

    # Primera separación: Detectamos palabras pegadas usando expresiones regulares.
    # Esto separa una minúscula seguida por una mayúscula.
    # Ejemplo: 'Privateresearchuniversity' -> 'Private Research University'.
    type_value = re.sub(r"([a-z])([A-Z])", r"\1 \2", type_value)

    # Capitalizamos todas las palabras (primera letra en mayúscula).
    # Ejemplo: 'public research' -> 'Public Research'.
    type_value = type_value.title()

    # Corrección específica para la palabra 'state', asegurándonos de que esté capitalizada.
    # Ejemplo: 'state-accredited university' -> 'State-Accredited University'.
    type_value = re.sub(r"\bState\b", "State", type_value, flags=re.IGNORECASE)

    # Revisión adicional para palabras clave no separadas correctamente (casos excepcionales).
    # Por ejemplo, 'Privateresearchuniversity' debería ser 'Private Research University'.
    # Este paso es más manual y se puede ampliar para otros casos específicos.
    if "Privateresearchuniversity" in type_value:
        type_value = type_value.replace("Privateresearchuniversity", "Private Research University")

    # Eliminamos espacios redundantes al principio o al final del texto.
    type_value = type_value.strip()

    # Devolvemos el texto limpio.
    return type_value

# Aplicamos la función a la columna 'type'.
# Cada valor de esta columna será procesado por la función clean_type_column.
df['type'] = df['type'].apply(clean_type_column)

# Nos aseguramos de que la columna 'type' contenga valores de tipo string.
df['type'] = df['type'].astype(str)

___

## 8.2.2. LIMPIEZA DE LA VARIABLE ESTABLISHED/ACTIVE (AÑO DE FUNDACIÓN DE LA UNIVERSIDAD)

In [185]:
def extract_year(date_value):
    """
    Extrae el año de un texto en la columna 'established/active' y lo convierte en un número entero.

    Si no se encuentra un año válido, devuelve None.

    Parámetros:
        date_value (str | None): El valor de la columna 'established/active'.
                                 Puede ser un texto con una fecha o un valor nulo.

    Retorno:
        int | None: El año extraído como un número entero, o None si no se encuentra un año.

    Ejemplo:
        Entrada: 'Founded in 1948; 75 years ago'
        Salida: 1948

        Entrada: 'Active since January 1, 1975'
        Salida: 1975

        Entrada: 'No applicable'
        Salida: None
    """

    # Si el valor está vacío o es nulo, devolvemos None.
    # Ejemplo:
    # - Entrada: None -> Salida: None
    # - Entrada: '' -> Salida: None
    if pd.isnull(date_value) or date_value == '':
        return None

    # Usamos una expresión regular para buscar un año (4 dígitos consecutivos).
    # Patrón: '\d{4}' busca cualquier número de 4 dígitos.
    # Ejemplo:
    # - Entrada: 'Founded in 1948' -> Coincidencias: ['1948']
    # - Entrada: 'Active since 1975; established earlier' -> Coincidencias: ['1975']
    year_matches = re.findall(r'\d{4}', date_value)

    # Si encontramos al menos un año, devolvemos el primero convertido a entero.
    # Ejemplo:
    # - Coincidencias: ['1948', '1975'] -> Salida: 1948
    if len(year_matches) > 0:
        return int(year_matches[0])

    # Si no encontramos ningún año, devolvemos None.
    # Ejemplo:
    # - Entrada: 'No applicable' -> Salida: None
    return None

# Aplicamos la función extract_year a la columna 'established/active'.
# Esto transforma cada valor de texto en un año (entero) o None si no hay datos válidos.
# Ejemplo:
# - Antes: 'Founded in 1948; 75 years ago'
# - Después: 1948
df['established/active'] = df['established/active'].apply(extract_year)

___

## 8.2.3. LIMPIEZA DE LA VARIABLE BUDGET (PRESUPUESTO)

In [188]:
def clean_budget_column(budget_value):
    """
    Limpia y transforma los valores de la columna 'budget/endowment'.
    Convierte el presupuesto a millones de euros (float) y elimina información extra.

    Parámetros:
        budget_value (str | None): El valor de la columna 'budget/endowment'.
                                   Puede ser un texto que contiene el presupuesto
                                   o un valor nulo (None/NaN).

    Retorno:
        float | None: El presupuesto en millones de euros, convertido a un número flotante.
                      Si no se encuentra un valor válido, devuelve None.

    Ejemplo de Procesamiento:
        Entrada: "€648.8 million" -> Salida: 648.8
        Entrada: "€1.2 billion" -> Salida: 1200.0
        Entrada: "No applicable" -> Salida: None
    """

    # Si el valor está vacío o es nulo, devolvemos None.
    # Ejemplo:
    # - Entrada: None -> Salida: None
    # - Entrada: '' -> Salida: None
    if pd.isnull(budget_value) or budget_value == '':
        return None

    # Variable para determinar si el valor está en millones o miles de millones (billones americanos).
    # Inicializamos como millón (factor multiplicador = 1).
    # Ejemplo:
    # - Entrada: "€648.8 million" -> is_million = 1
    # - Entrada: "€1.2 billion" -> is_million = 1000
    is_million = 1

    # Detectamos si el texto incluye "billion" (billones americanos, miles de millones).
    # En este caso, ajustamos el factor multiplicador a 1000.
    if "billion" in budget_value.lower():
        is_million = 1000

    # Usamos una expresión regular para extraer el número del texto.
    # Patrón: '\d+\.?\d+' encuentra números enteros o decimales.
    # Ejemplo:
    # - Entrada: "€648.8 million" -> Coincidencias: ['648.8']
    # - Entrada: "€1.2 billion" -> Coincidencias: ['1.2']
    number_matches = re.findall(r'\d+\.?\d+', budget_value)

    # Si encontramos un número válido, lo convertimos a float y ajustamos según el factor.
    # Ejemplo:
    # - Coincidencias: ['648.8'], is_million: 1 -> Salida: 648.8
    # - Coincidencias: ['1.2'], is_million: 1000 -> Salida: 1200.0
    if len(number_matches) > 0:
        return float(number_matches[0]) * is_million

    # Si no se encuentra un número válido, devolvemos None.
    # Ejemplo:
    # - Entrada: "No applicable" -> Salida: None
    return None

# Aplicamos la función de limpieza a la columna 'budget/endowment'.
# Esto transforma cada valor de texto en un presupuesto en millones de euros.
# Ejemplo:
# - Antes: "€648.8 million"
# - Después: 648.8
df['budget/endowment'] = df['budget/endowment'].apply(clean_budget_column)

# Renombramos la columna a 'Budget (million €)' para reflejar el formato de los valores.
# Esto actualiza el nombre de la columna para que sea claro que el presupuesto está en millones de euros.
# Ejemplo:
# - Antes: "budget/endowment"
# - Después: "Budget (million €)"
df.rename(columns={'budget/endowment': 'Budget (million €)'}, inplace=True)


---
## 8.2.4. LIMPIEZA DE LA VARIABLE PRESIDENT/RECTOR

In [191]:
def clean_president_column(president_value):
    """
    Limpia y uniformiza los valores de la columna 'president/rector'.

    Operaciones realizadas:
    1. Elimina referencias como '[1]', '[2]', '[de]', etc., incluso si están pegadas al nombre.
    2. Elimina tratamientos honoríficos como 'Prof. Dr.' o 'Dr.'. Esto se realiza para uniformizar este campo, ya que para muchas de las universidades no tenemos estos tratamientos documentados en Wikipedia.
    3. Elimina cualquier texto adicional entre paréntesis o corchetes inmediatamente después del apellido.
    4. Quita espacios en blanco adicionales al principio o al final del texto.

    Parámetros:
        president_value (str | None): El valor de la columna 'president/rector'.
                                      Puede ser un texto con el nombre del presidente/rector o un valor nulo.
                                      Lo hacemos por protección frente a errores, aunque en principio los nulos han sido sustituidos por
                                      "No applicable" en el Crawler.

    Retorno:
        str | None: El nombre limpio y uniforme. Si no hay información válida, retorna None.

    Ejemplo de Procesamiento:
        Entrada: "Prof. Dr. Günter M. Ziegler(2018–present)[1]"
        Salida: "Günter M. Ziegler"

        Entrada: "Elizabeth Prommer(907th rector)"
        Salida: "Elizabeth Prommer"

        Entrada: "No applicable"
        Salida: None
    """

    # Si el valor está vacío o es nulo, devolvemos None.
    # Ejemplo:
    # - Entrada: None -> Salida: None
    # - Entrada: '' -> Salida: None
    if pd.isnull(president_value) or president_value == '':
        return None

    # Eliminamos referencias como '[1]', '[2]', '[de]', etc., incluso si están pegadas al nombre.
    # Patrón: '\[.*?\]' elimina cualquier texto dentro de corchetes '[...]'.
    # Ejemplo:
    # - Entrada: "Günter M. Ziegler[1]" -> Salida: "Günter M. Ziegler"
    # - Entrada: "Elizabeth Prommer[de]" -> Salida: "Elizabeth Prommer"
    president_value = re.sub(r'\[.*?\]', '', president_value)

    # Eliminamos tratamientos honoríficos como 'Prof. Dr.' o 'Dr.'.
    # Patrón: '(Prof\. Dr\.|Dr\.)' busca las cadenas exactas 'Prof. Dr.' o 'Dr.'.
    # Ejemplo:
    # - Entrada: "Prof. Dr. Günter M. Ziegler" -> Salida: "Günter M. Ziegler"
    president_value = re.sub(r'(Prof\. Dr\.|Dr\.)', '', president_value, flags=re.IGNORECASE)

    # Eliminamos cualquier texto adicional entre paréntesis inmediatamente después del apellido.
    # Patrón: '\(.*?\)' elimina cualquier texto dentro de paréntesis '(...)'.
    # Ejemplo:
    # - Entrada: "Elizabeth Prommer(907th rector)" -> Salida: "Elizabeth Prommer"
    # - Entrada: "Thomas Hofmann(list of presidents)" -> Salida: "Thomas Hofmann"
    president_value = re.sub(r'\(.*?\)', '', president_value)

    # Eliminamos espacios redundantes al principio y al final.
    # Ejemplo:
    # - Entrada: " Günter M. Ziegler " -> Salida: "Günter M. Ziegler"
    president_value = president_value.strip()

    # Devolvemos el valor limpio.
    # Ejemplo:
    # - Entrada: "Prof. Dr. Günter M. Ziegler(2018–present)[1]" -> Salida: "Günter M. Ziegler"
    return president_value

# Aplicamos la función de limpieza a la columna 'president/rector'.
# Esto transforma cada valor de texto en un nombre limpio y uniforme.
# Ejemplo:
# - Antes: "Prof. Dr. Günter M. Ziegler(2018–present)[1]"
# - Después: "Günter M. Ziegler"
df['president/rector'] = df['president/rector'].apply(clean_president_column)

# Nos aseguramos de que la columna 'president/rector' contenga valores de tipo string.
df['president/rector'] = df['president/rector'].astype(str)

---
## 8.2.5. LIMPIEZA DE LA VARIABLE STUDENTS (NÚMERO DE ESTUDIANTES)

In [194]:
def clean_students_column(students_value):
    """
    Limpia y procesa los valores de la columna 'students/postgraduates'.

    Operaciones realizadas:
    1. Interpreta puntos (.) como separadores de miles y los elimina.
    2. Elimina comas (,) también utilizadas como separadores.
    3. Extrae el número de estudiantes como entero.
    4. Elimina cualquier texto adicional como "approx.", "students", etc.
    5. Si no se encuentra ningún número, devuelve None.

    Parámetros:
        students_value (str | None): El valor de la columna 'students/postgraduates'.
                                     Puede ser un texto con información sobre los estudiantes o un valor nulo.

    Retorno:
        int | None: El número de estudiantes extraído como un entero.
                    Si no hay información válida, devuelve None.

    Ejemplo de Procesamiento:
        Entrada: "8.543 students (2022)"
        Salida: 8543

        Entrada: "12,516 estudiantes"
        Salida: 12516

        Entrada: "No applicable"
        Salida: None
    """

    # Si el valor está vacío o es nulo, devolvemos None.
    # Ejemplo:
    # - Entrada: None -> Salida: None
    # - Entrada: '' -> Salida: None
    if pd.isnull(students_value) or students_value == '':
        return None

    # Reemplazamos los puntos (.) por nada, para tratarlos como separadores de miles.
    # Ejemplo:
    # - Entrada: "8.543 students" -> Salida intermedia: "8543 students"
    students_value = students_value.replace('.', '')

    # Reemplazamos las comas (,) por nada, para unificar el formato numérico.
    # Ejemplo:
    # - Entrada: "12,516 estudiantes" -> Salida intermedia: "12516 estudiantes"
    students_value = students_value.replace(',', '')

    # Usamos una expresión regular para encontrar el número en el texto.
    # Patrón: '\d+' busca números enteros.
    # Ejemplo:
    # - Entrada: "8543 students" -> Coincidencias: ['8543']
    # - Entrada: "No applicable" -> Coincidencias: []
    number_matches = re.findall(r'\d+', students_value)

    # Si encontramos al menos un número, lo convertimos a entero.
    # Ejemplo:
    # - Coincidencias: ['8543'] -> Salida: 8543
    if len(number_matches) > 0:
        return int(number_matches[0])

    # Si no encontramos ningún número, devolvemos None.
    # Ejemplo:
    # - Entrada: "No applicable" -> Salida: None
    return None

# Aplicamos la función de limpieza a la columna 'students/postgraduates'.
# Esto transforma cada valor de texto en un número entero representando la cantidad de estudiantes.
# Ejemplo:
# - Antes: "8.543 students (2022)"
# - Después: 8543
df['students/postgraduates'] = df['students/postgraduates'].apply(clean_students_column)

# Convertimos la columna 'students/postgraduates' explícitamente a tipo entero, manejando los valores nulos como NaN.
# Esto asegura que los datos faltantes se representen correctamente en el DataFrame.
# Ejemplo:
# - Antes: [8543, 12516, None]
# - Después: [8543, 12516, <NA>]
df['students/postgraduates'] = df['students/postgraduates'].astype('Int64') 

---
## 8.2.6. LIMPIEZA DE LA VARIABLE WEB

Para la variable Web, no es necesario realizar una limpieza ya que el formato obtenido con el crawler es el correcto. Tan solo realizaremos la corrección a string.

In [198]:
# Nos aseguramos de que la columna 'website' contenga valores de tipo string.
df['website'] = df['website'].astype(str)

---
## 8.2.7. MODIFICACIÓN DE LOS HEADERS DEL DATA FRAME Y ELIMINACIÓN DE LA PRIMERA COLUMNA

In [201]:
# Eliminamos la columna de file_name, que teníamos provisionalmente para encontrar fácilmente el html en el caso de detectar un error 
# en el csv
df = df.iloc[:, 1:]

# Con fines estéticos, modificamos los nombres de las columnas. A continuación se explica detalladamente lo que hacemos:
# 
# Creamos una nueva lista de nombres de columnas aplicando las siguientes reglas:
# - Poner la primera letra de cada palabra en mayúscula (usando `.capitalize()`).
# - Reemplazar "students/postgraduates" por "Students".
# - Reemplazar "budget/endowment" por "Budget".
# - Reemplazar "established/active" por "Established".
# - Reemplazar "president/rector" por "President / Rector".
new_column_names = [
    col.capitalize() if col not in ["students/postgraduates", "budget/endowment", "established/active", "president/rector"] else
    ("Students" if col == "students/postgraduates" else
     "Budget" if col == "budget/endowment" else
     "Established" if col == "established/active" else
     "President / Rector")
    for col in df.columns
]

# Asignamos los nuevos nombres al DataFrame.
df.columns = new_column_names


---
## 8.2.7. ESCRIBIMOS LOS RESULTADOS LIMPIOS

In [204]:
# Guardamos los datos procesados en un nuevo archivo CSV.
# float_format='%1f': Asegura que los valores de tipo float ('Budget' en nuestro caso) se guarden con un formato consistente 
# (por ejemplo, dos decimales).

df.to_csv(output_csv, index=False, float_format='%.1f')

print(f"Archivo limpio guardado en: {output_csv}")

Archivo limpio guardado en: universities_data_cleaned.csv
